In [1]:
from typing import TypedDict, List
from langchain_core.messages import BaseMessage

class ResearchAgentState(TypedDict):
    messages: List[BaseMessage]
    research_query: str
    search_results: List[str]
    analysis: str
    final_response: str

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

# Initialize LLM
llm = ChatOllama(model="qwen2.5:7b", base_url="http://localhost:11434", temperature=0.2)

def query_analysis_node(state: ResearchAgentState):
    """Analyze user query and extract research topics"""
    user_message = state["messages"][-1].content
    
    analysis_prompt = f"""
    Analyze this research request and extract the main topics to investigate:
    Request: {user_message}
    
    Provide 3-5 specific search queries that would gather comprehensive information.
    """
    
    response = llm.invoke([HumanMessage(content=analysis_prompt)])
    
    return {
        **state,
        "research_query": response.content,
        "messages": state["messages"] + [AIMessage(content=f"Research plan: {response.content}")]
    }

def search_node(state: ResearchAgentState):
    """Simulate research search (replace with actual search API)"""
    # In production, integrate with search APIs like Tavily, SerpAPI, or custom search
    mock_results = [
        "Recent developments in the field show significant progress...",
        "Key industry experts have noted the following trends...",
        "Statistical data indicates a 40% increase over the past year..."
    ]
    
    return {
        **state,
        "search_results": mock_results
    }

def analysis_node(state: ResearchAgentState):
    """Analyze search results and synthesize insights"""
    results_text = "\n".join(state["search_results"])
    
    analysis_prompt = f"""
    Based on these research results, provide a comprehensive analysis:
    
    Results:
    {results_text}
    
    Original query: {state["research_query"]}
    
    Provide key insights, trends, and actionable conclusions.
    """
    
    response = llm.invoke([HumanMessage(content=analysis_prompt)])
    
    return {
        **state,
        "analysis": response.content
    }

def response_generation_node(state: ResearchAgentState):
    """Generate final response for user"""
    final_prompt = f"""
    Create a comprehensive response based on this analysis:
    
    Analysis: {state["analysis"]}
    Original request: {state["messages"][0].content}
    
    Structure your response with:
    1. Executive summary
    2. Key findings
    3. Detailed insights
    4. Recommendations
    """
    
    response = llm.invoke([HumanMessage(content=final_prompt)])
    
    return {
        **state,
        "final_response": response.content,
        "messages": state["messages"] + [AIMessage(content=response.content)]
    }

In [3]:
from langgraph.graph import StateGraph, END

def build_research_agent():
    """Construct the research agent workflow"""
    graph = StateGraph(ResearchAgentState)
    
    # Add nodes to the graph
    graph.add_node("analyze_query", query_analysis_node)
    graph.add_node("search", search_node)
    graph.add_node("analyze_results", analysis_node)
    graph.add_node("generate_response", response_generation_node)
    
    # Define the workflow edges
    graph.set_entry_point("analyze_query")
    graph.add_edge("analyze_query", "search")
    graph.add_edge("search", "analyze_results")
    graph.add_edge("analyze_results", "generate_response")
    graph.add_edge("generate_response", END)
    
    return graph.compile()

# Create the agent
research_agent = build_research_agent()

In [4]:
from langchain_core.messages import HumanMessage

def test_research_agent():
    """Test the research agent with a sample query"""
    initial_state = {
        "messages": [HumanMessage(content="What are the latest trends in renewable energy adoption?")],
        "research_query": "",
        "search_results": [],
        "analysis": "",
        "final_response": ""
    }
    
    # Run the agent
    result = research_agent.invoke(initial_state)
    
    print("Final Response:")
    print("=" * 50)
    print(result["final_response"])
    
    return result

# Execute test
test_result = test_research_agent()

Final Response:
### Comprehensive Analysis on Latest Trends in Renewable Energy Adoption

#### 1. Executive Summary
The latest trends in renewable energy adoption are characterized by significant growth in solar and wind technologies, driven by technological advancements and declining costs. Offshore wind is emerging as a key player due to its higher capacity factors and growing economies of scale. Enhanced grid integration through smart grids and battery storage solutions is facilitating the seamless incorporation of intermittent renewable sources into existing power networks. International collaboration and supportive policies are fostering global cooperation and knowledge sharing, while environmental considerations continue to shape sustainable practices in the industry.

#### 2. Key Findings
- **Types of Renewable Energy**: Solar energy continues to dominate due to its widespread availability and decreasing costs, with recent advancements including bifacial solar panels and floatin

In [5]:
!pip install rich langfuse openai


  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.10.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/951.0 kB ? eta -:--:--
   --------------------------------------- 951.0/951.0 kB 14.7 MB/s eta 0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.10.0-cp312-cp312-win_amd64.whl (206 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# # LANGFUSE_SECRET_KEY=sk-lf-**********
# # LANGFUSE_PUBLIC_KEY=pk-lf-**********
# # LANGFUSE_HOST=http://10.46.41.225:3000


# langfuse = Langfuse(
#     public_key=os.getenv("pk-lf-*****************"),
#     secret_key=os.getenv("sk-lf-*****************"),
#     host=os.getenv("http://0.0.0.0:3000")
# )

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "gpt-oss",
#     "ollama",
#     "rich",
# ]
# ///
import random

from rich import print
# from langfuse.openai import OpenAI
from ollama import Client
from ollama._types import ChatResponse


def get_weather(city: str) -> str:
  """
  Get the current temperature for a city

  Args:
      city (str): The name of the city

  Returns:
      str: The current temperature
  """
  temperatures = list(range(-10, 35))

  temp = random.choice(temperatures)

  return f'The temperature in {city} is {temp}°C'


def get_weather_conditions(city: str) -> str:
  """
  Get the weather conditions for a city

  Args:
      city (str): The name of the city

  Returns:
      str: The current weather conditions
  """
  conditions = ['sunny', 'cloudy', 'rainy', 'snowy', 'foggy']
  return random.choice(conditions)


available_tools = {'get_weather': get_weather, 'get_weather_conditions': get_weather_conditions}

# messages = [{'role': 'user', 'content': 'What is the weather like in London? What are the conditions in Toronto?'}]
messages = [{'role': 'user', 'content': 'is 10 prime number?'}]


client = Client(
  host="http://localhost:11434",
)

model="qwen3:4b"

while True:
  response: ChatResponse = client.chat(model=model, messages=messages, tools=[get_weather, get_weather_conditions], think=False)

  if response.message.content:
    print('Content: ')
    print(response.message.content + '\n')
  if response.message.thinking:
    print('Thinking: ')
    print(response.message.thinking + '\n')

  messages.append(response.message)

  if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
      function_to_call = available_tools.get(tool_call.function.name)
      if function_to_call:
        result = function_to_call(**tool_call.function.arguments)
        print('Result from tool call name: ', tool_call.function.name, 'with arguments: ', tool_call.function.arguments, 'result: ', result + '\n')
        messages.append({'role': 'tool', 'content': result, 'tool_name': tool_call.function.name})
      else:
        print(f'Tool {tool_call.function.name} not found')
        messages.append({'role': 'tool', 'content': f'Tool {tool_call.function.name} not found', 'tool_name': tool_call.function.name})
  else:
    # no more tool calls, we can stop the loop
    break

In [16]:
print(response)

ChatResponse(
    model='qwen3:4b',
    created_at='2025-09-10T14:01:15.8182123Z',
    done=True,
    done_reason='stop',
    total_duration=46308327200,
    load_duration=98359300,
    prompt_eval_count=204,
    prompt_eval_duration=615443200,
    eval_count=256,
    eval_duration=45592239400,
    message=Message(
        role='assistant',
        content='<think>\nOkay, let\'s see. The user is asking if 10 is a prime number. Hmm, prime numbers are 
numbers greater than 1 that have no divisors other than 1 and themselves. Wait, 10... let me think. 10 divided by 2
is 5, so 2 and .5? No, 10 is even, so it\'s divisible by 2. So 10 factors into 2 and 5. That means it\'s not a 
prime number. But wait, the tools provided here are for getting weather data. The functions are get_weather and 
get_weather_conditions, both require a city name. The user\'s question is about math, not weather. So there\'s no 
relevant function here to call. I should inform them that I can\'t help with that using the available 
tools.\n</think>\n\nThe question "is 10 prime number?" is a mathematical query, but none of the provided tools 
(which are weather-related functions) can assist with this. I cannot use the available tools to answer this 
question. \n\n**Answer:** No, 10 is not a prime number because it can be divided evenly by 2 and 5 (other than 1 
and itself).',
        thinking=None,
        images=None,
        tool_name=None,
        tool_calls=None
    )
)